# Mental Health Text Risk Detector (DistilBERT)

This notebook fine-tunes a `distilbert-base-uncased` model to predict mental health risks based on raw Reddit data. Training a transformer model is resource-intensive. For best results, run this on a machine with a dedicated GPU (e.g., Google Colab with T4/A100).

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

## Data Loading
Load data from the raw directory, mapping filenames to labels. We sample 2000 rows per class.

In [ ]:
raw_data_dir = 'Dataset/Original Reddit Data/raw data/'
all_files = glob.glob(os.path.join(raw_data_dir, '**', '*.csv'), recursive=True)
dfs = []
for f in all_files:
    df = pd.read_csv(f)
    if 'selftext' not in df.columns:
        continue
    
    fname = os.path.basename(f).lower()
    if 'dep' in fname:
        label = 'Depression'
    elif 'ani' in fname or 'anx' in fname:
        label = 'Anxiety'
    elif 'lone' in fname:
        label = 'Loneliness'
    elif 'mh' in fname:
        label = 'Mental Health'
    elif 'sw' in fname or 'suicide' in fname:
        label = 'High Risk (SW)'
    else:
        label = 'Other'
        
    df['Label'] = label
    # Sample up to 2000 rows per file to avoid memory issues
    df_sampled = df.dropna(subset=['selftext']).sample(min(2000, len(df.dropna(subset=['selftext']))), random_state=42)
    dfs.append(df_sampled[['selftext', 'Label']])

train_df = pd.concat(dfs, ignore_index=True)
print("Training data size:", len(train_df))
print(train_df['Label'].value_counts())

## Preprocessing & Feature Extraction
Clean the text and set up ID mapping for the classification head.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    return text

train_df['cleaned_text'] = train_df['selftext'].apply(clean_text)
train_df = train_df[train_df['cleaned_text'].str.strip() != ""]

labels = train_df['Label'].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

train_df['label_id'] = train_df['Label'].map(label2id)
print(f"Labels mapping: {label2id}")

## Model Setup & Tokenization
Initialize Hugging Face Tokenizer and prepare the dataset.

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create HF Dataset
hf_dataset = Dataset.from_pandas(train_df[['cleaned_text', 'label_id']])
if '__index_level_0__' in hf_dataset.column_names:
    hf_dataset = hf_dataset.remove_columns(['__index_level_0__'])
hf_dataset = hf_dataset.rename_column("label_id", "labels")
hf_dataset = hf_dataset.rename_column("cleaned_text", "text")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

print("Tokenizing dataset...")
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
train_testvalid = tokenized_datasets.train_test_split(test_size=0.1)

## Training
Fine-tune the model.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(labels), 
    id2label=id2label, 
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_testvalid["train"],
    eval_dataset=train_testvalid["test"],
    processing_class=tokenizer,
)

trainer.train()

## Save Model
Save the model so Streamlit (`app.py`) can use it.

In [ ]:
output_dir = 'models/distilbert'
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")